In [ ]:
import pandas as pd
import numpy as np
import dai

def main(datasources, start_date, end_date):
    bar1m_table = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")  # 高频, 仅盘口因子用
    financial_table = datasources.get("financial", "bigalpha_2026_financial")
    bar1d_table = "bigalpha_2026_bar1d"                   # 日频, 优先使用
    factorlib_table = "bigalpha_2026_factorlib"           # 日频预计算
    exposure_table = "bigalpha_2026_exposure"             # 日频暴露

    start_dt = pd.Timestamp(start_date)
    end_dt = pd.Timestamp(end_date)
    calc_start = start_dt - pd.Timedelta(days=360)  # max(20, 4, 2) = 20天回看，360足够

    # ★ 数据读取 (dai.query) ★
    df = dai.query(f"SELECT date, instrument, daily_return, amount, close FROM {factorlib_table}", filters={'date': [str(calc_start), str(end_date)]}).df()
    
    # 阶段1: 数据输入检查
    if df.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])
    assert all(c in df.columns for c in ['date', 'instrument']), "缺少必需列 date 或 instrument"
    df['date'] = pd.to_datetime(df['date'])
    df = df.drop_duplicates(['date', 'instrument'])
    
    # 阶段2: 数据清洗
    for col in ['daily_return', 'amount', 'close']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df.groupby('instrument')[col].ffill()
            df[col] = df[col].fillna(0)
    
    # ★ 因子计算 (pandas/numpy) ★
    # 按股票分组，计算所需序列
    df = df.sort_values(['instrument', 'date']).reset_index(drop=True)
    
    # rank(-1 * daily_return * amount)
    df['tmp1'] = -1 * df['daily_return'] * df['amount']
    # 阶段3: 分组计算检查 - rank 前确保 groupby('date')
    df['rank1'] = df.groupby('date')['tmp1'].rank(pct=True)
    
    # rank(mean(amount, 2) / (1e-8 + std(amount, 4)))
    # 阶段3: rolling 前确保 groupby('instrument')
    df['mean_amount_2'] = df.groupby('instrument')['amount'].rolling(2).mean().reset_index(level=0, drop=True)
    df['std_amount_4'] = df.groupby('instrument')['amount'].rolling(4).std().reset_index(level=0, drop=True)
    # 阶段4: 异常保护 - 数学运算保护
    try:
        df['tmp2'] = df['mean_amount_2'] / (1e-8 + df['std_amount_4'])
    except Exception:
        df['tmp2'] = np.nan
    df['tmp2'] = df['tmp2'].replace([np.inf, -np.inf], np.nan)
    df['rank2'] = df.groupby('date')['tmp2'].rank(pct=True)
    
    # rank(-1 * sum(daily_return, 2))
    df['sum_daily_return_2'] = df.groupby('instrument')['daily_return'].rolling(2).sum().reset_index(level=0, drop=True)
    df['tmp3'] = -1 * df['sum_daily_return_2']
    df['rank3'] = df.groupby('date')['tmp3'].rank(pct=True)
    
    # rank(1 - (close / max(close, 20)))
    df['max_close_20'] = df.groupby('instrument')['close'].rolling(20).max().reset_index(level=0, drop=True)
    # 阶段4: 异常保护 - 除法保护
    try:
        df['tmp4'] = 1 - (df['close'] / df['max_close_20'])
    except Exception:
        df['tmp4'] = np.nan
    df['tmp4'] = df['tmp4'].replace([np.inf, -np.inf], np.nan)
    df['rank4'] = df.groupby('date')['tmp4'].rank(pct=True)
    
    # 因子 = rank1 * rank2 * rank3 * rank4
    df['factor'] = df['rank1'] * df['rank2'] * df['rank3'] * df['rank4']

    result = df[['date', 'instrument', 'factor']]
    # 阶段5: 最终输出
    result['factor'] = result['factor'].fillna(0)
    result = result.dropna(subset=['factor'])
    # 阶段6: BLAS/LAPACK 安全 - dropna 前检查
    if not result.empty:
        result = result.dropna(subset=['factor'])
    result = result[result['date'] >= str(start_dt)]
    return result